# 从零开始构建推理引擎（一）：引入 KV Cache 基础优化

## 1. 回顾与问题分析

在上一章中，我们实现了一个朴素的 Qwen3 推理流程。
其核心循环逻辑如下：

1. 输入 `[T1, T2, T3]`
2. 模型计算，输出 `T4`
3. 下一轮输入 `[T1, T2, T3, T4]`
4. 模型计算，输出 `T5`
5. 依此类推

这里存在一个巨大的**计算浪费**。
从直觉上来看，如果模型保存了第二步中对 `[T1, T2, T3]` 的中间计算结果，
是否可以在第四步中继续利用这些结果呢？
打个比方，如果对输入序列的计算逻辑为求和，初始输入序列为 `[1, 3, 2]`，
这一轮的计算得到求和结果为 `6`，并保存起来。
在新一轮求和时，输入序列变为 `[1, 3, 2, 5]`，我们可以避免再去计算 `[1, 3, 2]` 的求和结果，
直接从保存的计算结果 `6` 加上 `5`，得到 `11`。

纵观整个模型的计算过程，是否存在这样的优化点呢？
我们在上一章已经很清楚 Qwen3 这样的模型的计算结构，这边再列出作为参考。

```plaintext
Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
      )
    )
    (norm): Qwen3RMSNorm((1024,), eps=1e-06)
    (rotary_emb): Qwen3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=1024, out_features=151936, bias=False)
)
```

实际上计算浪费的出现点只和那些会对**整个序列**进行计算的模块相关。
在 Qwen3 模型中，也只有 `Qwen3Attention` 模块会对整个序列进行计算。
这就是我们接下来的优化点。

## 2. 优化方案

